# Train SmolVLA — SO-101 Pick & Place (by color)

Finetune **SmolVLA** บน dataset ที่เก็บด้วย `record_scripted.py` (LeRobotDataset v3.0)
เพื่อสั่งหุ่นหยิบลูกบอลตามสีด้วยภาษา

**ใช้กับ Colab GPU** (เครื่อง Mac ไม่มี CUDA → เก็บ dataset บน Mac, push ขึ้น HF Hub, มา train ที่นี่)

> Runtime → Change runtime type → **GPU** (T4 พอสำหรับ batch เล็ก, A100 ดีสุด)

## 1. ติดตั้ง lerobot + SmolVLA

In [ ]:
!pip install -q 'lerobot[smolvla]'
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

## 2. Login HuggingFace (เพื่อดึง dataset + smolvla_base)

In [ ]:
from huggingface_hub import login
login()  # ใส่ token (มีสิทธิ์อ่าน dataset ของคุณ)

## 3. ตั้งค่า + train

แก้ `HF_USER` และ `DATASET` ให้ตรงกับ dataset ที่ push ไว้

In [ ]:
HF_USER = 'your-hf-username'          # << แก้
DATASET = f'{HF_USER}/so101_pickplace_color'
STEPS = 20000                          # dataset ใหญ่ขึ้น → เพิ่มเป็น 50k-100k
BATCH = 64                             # ลดถ้า VRAM ไม่พอ (เช่น 16)
print('dataset:', DATASET)

In [ ]:
# ตรวจ dataset ก่อน train (key ครบ + task strings)
from lerobot.datasets.lerobot_dataset import LeRobotDataset
d = LeRobotDataset(DATASET)
print('episodes:', d.num_episodes, 'frames:', d.num_frames)
print('features:', [k for k in d.features if k.startswith(('observation','action'))])
print('task example:', d[0].get('task'))

In [ ]:
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=$DATASET \
  --batch_size=$BATCH \
  --steps=$STEPS \
  --output_dir=outputs/smolvla_so101 \
  --policy.device=cuda

## 4. Push checkpoint กลับ HF Hub (เอาไป deploy บน Mac)

checkpoint สุดท้ายอยู่ที่ `outputs/smolvla_so101/checkpoints/last/pretrained_model`

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo = f'{HF_USER}/smolvla_so101_pickplace'
api.create_repo(repo, exist_ok=True)
api.upload_folder(folder_path='outputs/smolvla_so101/checkpoints/last/pretrained_model', repo_id=repo)
print('pushed:', repo)

## 5. Deploy (บน Mac)

```bash
# ดึง checkpoint จาก Hub มา deploy
python -m robot_learning.deploy.voice_pipeline \
  --checkpoint your-hf-username/smolvla_so101_pickplace \
  --text 'หยิบลูกบอลสีแดงใส่ถาด'
```